# 07_opaque — Exclude sensitive fields from state x-ray

By default, every field declared with `@result_*` appears in OTel Logs as an
`aoa.state.<field>` attribute. Add `opaque=True` to exclude a field from the x-ray.

`opaque=True` does **not** affect:
- The checker — the field is still validated normally
- `state["field"]` — the value is still in state, readable by downstream aspects
- OTel Traces — spans are unaffected

This notebook shows a payment step with two fields:
- `result_string("order_id")` — default, appears in logs
- `result_string("payment_token", opaque=True)` — absent from logs

In [ ]:
!pip install "aoa-action-machine[otel]"

In [ ]:
import asyncio

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import ConsoleLogRecordExporter, SimpleLogRecordProcessor
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.open_telemetry import OpenTelemetryPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params, Result

In [ ]:
class PaymentDomain(BaseDomain):
    name = "payment"
    description = "Payment processing domain"


class PaymentParams(BaseParams):
    order_id: str = Field(description="Order identifier")
    card_last4: str = Field(description="Last four digits of the card")


class PaymentResult(BaseResult):
    order_id: str = Field(description="Processed order ID")
    masked_token: str = Field(description="Masked payment token for the receipt")

## Action — opaque=True on the payment token

The `payment_token` field is checked, stored in state, and used by the summary.
But it will not appear in any `aoa.state.*` log attribute.

In [ ]:
@meta(description="Charge order and store payment token", domain=PaymentDomain)
@check_roles(GuestRole)
class ChargeOrderAction(BaseAction[PaymentParams, PaymentResult]):

    @regular_aspect("Validate order and mint payment token")
    @result_string("order_id", required=True, min_length=3)
    @result_string("payment_token", required=True, opaque=True)  # excluded from x-ray
    async def payment_aspect(self, params, state, box, connections):
        token = f"tok_live_{params.card_last4}_{'x' * 16}"
        return {
            "order_id": params.order_id.upper(),
            "payment_token": token,
        }

    @summary_aspect("Build payment receipt")
    async def receipt_summary(self, params, state, box, connections):
        token: str = state["payment_token"]   # still available in state
        masked = token[:8] + "****" + token[-4:]
        return PaymentResult(order_id=state["order_id"], masked_token=masked)

## Run

Look at the `aoa.aspect.regular.after` log record for `payment_aspect`:
- `aoa.state.order_id` will appear
- `aoa.state.payment_token` will **not** appear

`payment_token` is still in state — the summary uses it to build the masked token.

In [ ]:
lp = LoggerProvider()
lp.add_log_record_processor(SimpleLogRecordProcessor(ConsoleLogRecordExporter()))

plugin = OpenTelemetryPlugin(logger_provider=lp, service_name="payment-service")
machine = ActionProductMachine(plugins=[plugin])

result = await machine.run(
    Context(),
    ChargeOrderAction(),
    PaymentParams(order_id="ord-2024-001", card_last4="4242"),
)
print(f"Result: order={result.order_id}, token={result.masked_token}")
print()
print("Check the 'aoa.aspect.regular.after' record above:")
print("  aoa.state.order_id      → present")
print("  aoa.state.payment_token → absent (opaque=True)")